# Tutorial: CrossRegistry

The `CrossRegistry` allows you to conveniently get, aggregate, and label data stored
at the CROSS data platform

## Packages and data

In [1]:
# to manage your .env file, you can use the python-dotenv package. 
# Install it with pip if you haven't already:
from dotenv import load_dotenv
import os

# Import the CrossRegistry class from the crosscontract package
from crosscontract import CrossRegistry

## Creating the CrossRegistry

To create the registry, you simply provide your username and password. Here we 
assume that your credentials are stored in a 
.env file and we extract them from there.

**Note** Do not store your credentials in GitHub!

In [2]:
# load the environment variables from the .env file
load_dotenv(".env")
username = os.getenv("CROSSUSER")

# create the registry using the environment variables
my_registry = CrossRegistry(
    username=os.getenv("CROSSUSER"), 
    password=os.getenv("PASSWORD")
)

## Getting a varible

To get a variable, you need to now the name of the contract. To get on overview 
over your available contracts, you can use the `contract_overview` property.

In [3]:
my_registry.contract_overview.query("name.str.startswith('result_')")

,name,title,description
7,result_electricity_consumption,Result submission - Electricity consumption,Electricity consumption as submitted from scen...
14,result_electricity_supply,Result submission - Electricity supply,Electricity supply as submitted from scenario ...
15,result_h2_fec,Result submission - Hydrogen final energy cons...,Hydrogen final energy consumption as submitted...
19,result_h2_supply,Result submission - Hydrogen supply,Hydrogen supply as submitted from scenario runs
24,result_methane_consumption,Result submission - Methane final energy consu...,Methane final energy consumption as submitted ...
27,result_methane_supply,Result submission - Methane supply,Methane supply as submitted from scenario runs
28,result_liquids_consumption,Result submission - Liquid fuels final energy ...,Liquid fuels final energy consumption as submi...
31,result_liquids_supply,Result submission - Liquid fuels supply,Liquid fuels supply as submitted from scenario...
32,result_process_heat_energy_production,Result submission - Process heat production,Useful energy production of process heat as su...
35,result_space_heat_energy_supply,Result submission - Space Heat supply,Useful energy supply of space heat as submitte...


Given the name,
you can add the variable to the registry or simply use dot notation. If you use
dot notation, the registry will automatically add the variable to the registry.

In [4]:
res_elec_supply = my_registry.result_electricity_supply
res_elec_supply

CrossDataVariable(name=result_electricity_supply, filters=None)

As you can see, we can provide a filter for the data. This is available if you use
the add() method and will filter the data already coming from the platform.

To add the variable again, we need to use `overwrite=True` as it is already in the 
registry. Here we filter to only get data for the year 2050. Note that this will
affect all later usage, as the filter is general, i.e., applied when the registry
fetches the data from the CROSS platform.

**Note** Currently server side filtering is rather restricted
- Only one value per field is allowd
- Only string columns can be filtered

In [5]:
res_elec_supply = my_registry.add_variable(
    "result_electricity_supply", 
    filters={"scenario_name": "abroad-res-full"}, 
    overwrite=True
)

## Assessing data

Now that you have the variable, you can access the data by using its `data` attribute.
Using the data attribute provides you the data stored at the platform as pandas
dataframe (with the filter already applied).

In [6]:
res_elec_supply.data.head()

,model,scenario_group,scenario_name,scenario_variant,technology,country,year,unit,value
0,powercheck,cross202506,abroad-res-full,reference,methane_chp_woccs,CH,2040,TWh,0.0
1,powercheck,cross202506,abroad-res-full,reference,methane_chp_woccs,CH,2050,TWh,0.0
2,powercheck,cross202506,abroad-res-full,reference,methane_chp_ccs,CH,2040,TWh,0.0
3,powercheck,cross202506,abroad-res-full,reference,methane_chp_ccs,CH,2050,TWh,0.0
4,powercheck,cross202506,abroad-res-full,reference,methane_oc_woccs,CH,2040,TWh,0.0


While the `.data` property provides access to the full dataset, the `get_data`
method allows you to specify additional filters, aggregate the data, and to label
items based on the information in the contract (and the references to the Cross Dimensions).

- **Filtering** is based on a dictionary with the key being the name of the column
and the value a list with the allowed values
- **Aggregation** is also dictionary based. The key is the name of the column over 
which to aggregate and the entry is an integer to specify the aggregation level. 0
is the highest aggregation level, i.e., the level with as little as possible details.
- **Labeling** is based on the `use_titles` parameter. If set to true all columns
will be automatically relabelled. 
- **Columns** allow to narrow the list of columns in the dataframe provided. Note that
the filter does not drop colums at all. Columns are always applied at the very end of
the transformation.

In [7]:
res_elec_supply.get_data(
    filters={"year": [2050], "scenario_variant": ["reference"]},
    aggregation={"technology": 0},
    use_titles=True,
    columns=["model", "year", "technology", "value"]
)

,model,year,technology,value
0,EHUB,2050,Electrochemical,4.674340
1,EHUB,2050,Imports of electricity,14.988541
2,EHUB,2050,Renewables,79.931771
3,EHUB,2050,Electricity storage,22.561954
4,EHUB,2050,Thermal power plants,1.222394
5,PowerCheck,2050,Electrochemical,0.222356
6,PowerCheck,2050,Imports of electricity,10.744632
7,PowerCheck,2050,Renewables,64.901922
8,PowerCheck,2050,Electricity storage,5.975591
9,PowerCheck,2050,Thermal power plants,3.771370


## Examine dimensions

The value variable we have in the registry might refer to the CROSS data model,
i.e., the dimensions as given the CROSS platform. It thus can be helpful to examine
these dimensions. If you have the value variable (e.g., our res_elec_supply variable),
there are two ways. First, you can reach the dimension through the registry given
its contract name. Second, and more convenient, you can access it through the variable
using the name of the column that refers to the dimension:

In [8]:
res_elec_supply.dimensions["technology"].data.head()

,id,level,id_parent,label,explanation
0,thermal,0,,Thermal power plants,
1,methane_pp,1,thermal,"Methane (includes natural gas, biogas, biometh...",
2,methane_chp,2,methane_pp,"Methane (includes natural gas, biogas, biometh...",
3,methane_chp_woccs,3,methane_chp,"Methane (includes natural gas, biogas, biometh...",
4,methane_chp_ccs,3,methane_chp,"Methane (includes natural gas, biogas, biometh...",
